In [18]:
from InputData import *

instance_filename = "AnzahlAuftraege_NEW_10/Construction_a10_o107_m5_an57_ar12.json"
#instance_filename = "AnzahlAuftraege_NEW_50/Construction_a50_o625_m30_an258_ar61.json"
instance_filename = "Construction_a1_o12_m3_an5_ar3_reduced.json"

instance_filename = "AnzahlAuftraege_NEW_50/Construction_a50_o639_m28_an294_ar66.json"

data = InputData(instance_filename)


In [19]:
# 2a. Sets
M = list()
W_m = dict()
N_m = dict()
for machine in data.machines:
    M.append(machine.id)
    W_m[machine.id] = [int(driver) for driver in machine.default_drivers]
    N_m[machine.id] = list()
    for orderItem in data.order_items:
        if orderItem.machine_type == machine.type:
            N_m[machine.id].append(orderItem.id)

W = list()
N_w = dict() # ANNAHME: N_w ist die Menge der Bestellungen, die ein Arbeiter bearbeiten kann
for worker in data.workers:
    W.append(worker.personal_number)
    for orderItem in data.order_items:
        if orderItem.worker_qualifications == []:
            if worker.personal_number not in N_w:
                N_w[worker.personal_number] = list()
            N_w[worker.personal_number].append(orderItem.id)
        elif orderItem.worker_qualifications == worker.qualifications:
            if worker.personal_number not in N_w:
                N_w[worker.personal_number] = list()
            N_w[worker.personal_number].append(orderItem.id)
        

'''
A = list()
A_Class = list()
N_a = dict() # ANNAHME: N_a ist die Menge der Bestellungen, die ein Anbaugerät bearbeiten kann
for attachment in data.attachments:
    A.append(attachment.id)
    A_Class.append(attachment.type)
'''
    
N = list()
for orderItem in data.order_items:
    N.append(orderItem.id)

C = list()
N_c = dict()
for order in data.orders:
    C.append(order.site_number)
    N_c[order.site_number] = [int(item_id) for item_id in order.order_item_ids]
print("C: ", C)


start_date = data.start_date
end_date = data.end_date

O_t = dict()  # Tag an dem der Auftrag startet
O_t_start = dict()  # Startzeiten  
O_t_end = dict()  # Endzeiten
O_t_start_inverted = dict()  # Umgekehrtes O_t (Startzeiten)
O_t_end_inverted = dict()  # Umgekehrtes O_t_end (Endzeiten)

SECONDS_IN_A_DAY = 86400

for orderItem in data.order_items:
    orderID = orderItem.id 

    # Startzeit
    orderItem_start_date = orderItem.start_time
    delta_start = (orderItem_start_date - start_date)
    t_start = delta_start.total_seconds() / SECONDS_IN_A_DAY
    t_start_int = int(t_start)


    # O_t: Gruppiert nach Tagen
    if t_start_int not in O_t:
        O_t[t_start_int] = []
    O_t[t_start_int].append(orderID)
    
    
    # Startzeit
    if t_start not in O_t_start:
        O_t_start[t_start] = []
    O_t_start[t_start].append(orderID)
    
    # Invertiertes Dictionary O_t_start_inverted
    O_t_start_inverted[orderID] = t_start

    # Endzeit
    orderItem_end_date = orderItem.end_time
    delta_end = (orderItem_end_date - start_date)
    t_end = delta_end.total_seconds() / SECONDS_IN_A_DAY

    # O_t_end: Gruppiert nach Endzeit
    if t_end not in O_t_end:
        O_t_end[t_end] = []
    O_t_end[t_end].append(orderID)
    
    # Invertiertes Dictionary O_t_end_inverted
    O_t_end_inverted[orderID] = t_end



P_mn = dict()
S_mn = dict()

d_ab = data.transport_routes
d_wb = data.work_routes
d_ij = list()
d_wj = list()

for i in data.order_items:
    row = []
    for j in data.order_items:
        a = next((k for k,v in N_c.items() if i.id in v))
        b = next((k for k,v in N_c.items() if j.id in v))
        row.append(d_ab[a][b])
    d_ij.append(row)


for i in data.workers:
    row = []
    for j in data.order_items:
        a = next((k for k,v in N_c.items() if j.id in v))
        row.append(d_wb[i.personal_number][a])
    d_wj.append(row)



SPEED = 1680 # Durchschnittliche Geschwindigkeit des Maschinentransports in 1680 km/Tag --> entspricht 70 km/h

start = len(N)
end = len(N) + 1

for m in M:
    for n in N_m[m]:


        if (m,start) not in P_mn:
            P_mn[m,start] = list()
            S_mn[m,start] = list()
            S_mn[m,start].append(end) # Anfügen Endknoten als Nachfolger des Startknotens
        if (m,end) not in P_mn:
            P_mn[m,end] = list()
            S_mn[m,end] = list()
            P_mn[m,end].append(start) # Anfügen Startknoten als Vorgänger des Endknotens


        P_mn[m,n] = list()
        S_mn[m,n] = list()


        P_mn[m,n].append(start) # Anfügen Startknoten als Vorgänger von n
        S_mn[m,start].append(n) # Anfügen n als Nachfolger des Startknotens

        P_mn[m,end].append(n) # Anfügen n als Vorgänger des Endknotens
        S_mn[m,n].append(end) # Anfügen Endknoten als Nachfolger von n
        
        for i in N_m[m]:
            if n != i:
                start_time_n = O_t_start_inverted[n]
                end_time_n = O_t_end_inverted[n]
                start_time_i = O_t_start_inverted[i]
                end_time_i = O_t_end_inverted[i]


                if start_time_n >= end_time_i: #+ d_ij[i][n] / SPEED: 
                    P_mn[m,n].append(i)

                if start_time_i > end_time_n:# + d_ij[n][i] / SPEED:
                    S_mn[m,n].append(i)


P_wn = dict()
S_wn = dict()

P_time = 9/24 # 9 Stunden Pausenzeit zwischen zwei Schichten --> 9/24 Tage

for w in W:
    for n in N_w[w]:
        
        if (w,start) not in P_wn:
            P_wn[w,start] = list()
            S_wn[w,start] = list()
            S_wn[w,start].append(end)
        if (w,end) not in P_wn:
            P_wn[w,end] = list()
            S_wn[w,end] = list()
            P_wn[w,end].append(start)
        
        P_wn[w,n] = list()
        S_wn[w,n] = list()

        P_wn[w,n].append(start)
        S_wn[w,start].append(n)
        
        P_wn[w,end].append(n)
        S_wn[w,n].append(end)
        
        for i in N_w[w]:
            if n != i:
                start_time_n = O_t_start_inverted[n]
                end_time_n = O_t_end_inverted[n]
                start_time_i = O_t_start_inverted[i]
                end_time_i = O_t_end_inverted[i]

                if start_time_n >= end_time_i:# + P_time:
                    P_wn[w,n].append(i)

                if start_time_i >= end_time_n:# + P_time:
                    S_wn[w,n].append(i)
        



day_difference = end_date - start_date
T_range = list(range(day_difference.days + 1))


# 2b. Parameter

T = day_difference.days + 1


S_Nmax = 5 # Maximal Anzahl an aufeinanderfolgenden Nachtschichten
S_max = 10 # Maximal Anzahl an Schichten im Zeitraum T_Smax
T_Smax = 14 # Zeitraum für S_max
T_Wmax = 40 # Maximale Arbeistzeit im Betrachtungszeitraum/Monat ?

t_o = list()
for orderItem in data.order_items:
    t_o.append(orderItem.duration)

C:  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]


In [20]:
W_m


{0: [34, 6, 21],
 1: [17, 11, 12],
 2: [12, 15, 8],
 3: [17, 49, 27],
 4: [1, 31, 15],
 5: [31, 51],
 6: [21, 10],
 7: [1, 41, 20],
 8: [31, 13],
 9: [20, 46, 17],
 10: [55, 45],
 11: [40, 1],
 12: [31, 35],
 13: [27, 10, 11],
 14: [6, 46],
 15: [62, 33],
 16: [65, 37],
 17: [11, 37],
 18: [4, 50],
 19: [54, 63, 3],
 20: [54, 11],
 21: [8, 47, 31],
 22: [42, 51],
 23: [31, 56],
 24: [57, 3],
 25: [5, 49, 52],
 26: [18, 9, 19],
 27: [62, 4]}

In [21]:
for w in W_m[0]:
    print(w)
    print(S_wn[w,0])

34
[640, 1, 2, 3, 4, 5, 6, 7, 110, 137, 138, 139, 140, 141, 142, 395, 396, 528, 583, 584, 585, 586, 587, 588, 615, 616, 617, 618, 619]
6
[640, 1, 2, 3, 4, 5, 6, 7, 110, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 183, 202, 217, 218, 219, 220, 221, 222, 252, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 312, 318, 319, 320, 321, 322, 365, 366, 368, 369, 370, 371, 372, 395, 396, 442, 443, 444, 445, 446, 459, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 528, 583, 584, 585, 586, 587, 588, 609, 615, 616, 617, 618, 619]
21
[640, 1, 2, 3, 4, 5, 6, 7, 110, 137, 138, 139, 140, 141, 142, 350, 351, 352, 395, 396, 528, 583, 584, 585, 586, 587, 588, 615, 616, 617, 618, 619]


In [22]:
# printe alle enzeiten der orderitems

# 17 uhr als datetime ohne datum



for orderItem in data.order_items:
    if orderItem.start_time.hour > 8 and orderItem.start_time.hour < 18:
        print(orderItem.id ,orderItem.start_time, orderItem.end_time, orderItem.duration)

In [26]:
print(*range(3,9))

3 4 5 6 7 8
